In [70]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.dataloader import create_dataloaderV0
from pathlib import Path

import torch
from torch import nn
import tiktoken

In [71]:
path = Path().cwd().parents[1] / "data" / "train_data.txt"

In [72]:
# load back the dataset
with open(path, 'r', encoding = 'utf-8') as f:
    text = f.read()

print(text[:1000])  # print the first 1000 characters of the dataset

Tahirah has been volunteering with Ethnic Minorities and Youth Support Team (EYST) Wales for over 5 years now, whilst contributing to the community and enjoying every minute of it. From play schemes to homework club, working with different community groups, she feels this has shaped her as a young individual.
In 2018, Tahirah became involved with Young, Migrant and Welsh (YMW), a project that focused on changing the perceptions of young individuals who live in Wales and are from a migrant community. Her contribution focused on females in sports, specifically representation and weightlifting.
Nominated as a Youth Ambassador for Wales by EYST Wales for the #iwill campaign Tahirah feels that she was able to reach a wider audience and raise more awareness, not only in the local community but nationally too. Whilst working with other Youth Ambassadors in Wales, Tahirah was able to contribute in creating material that reflected the diverse Wales.
Tahirah has also been involved in the judging

In [73]:
# now we will start implementing the multihead attention mechanism from scratch
# multihead attention is a key component of the transformer and the one that caused a major breakthrough in the field of NLP
# the idea behind it is to take in the input embeddings which only carry the meaning of the word itself and the position of them
# then we modify them enhancing them by adding the contextual information of the word in that sentence
# naming them "context vectors" denoted as Z
# kind of like moving the vector of the word in the embedding space a bit closer to the meaning of it in that context
# for example 
# i have walked across the river bank
# and 
# ill deposit the money in the bank
# without the attention mechanism the word "bank" will have the same embedding in both sentences
# and thus will be interpreted as the same word with the same meaning
# the idea behind it is by seeing the similarity of each word with all other words in that sequence
# and to calculate that similarity we use the dot product
# as the dot product shows or indicates how collinear two vectors are, and thus how similar they are
# the closer they are to one another the more similar they are, and the more they are pointing in the same direction
# the sequence goes as follows:
# first choose a word in the sentence we call it "query"
# second we take all the other words in the sentence and we call them "keys" think of them as the words that we are comparing the query to
# third we take the dot product of the query with all the keys to get a similarity score
# that similarity score is called "attention score" denoted as w
# then after we need to normalize the attention scores to make them more interpretable and to avoid numerical instability
# by doing so we can see them as probabilities, and we do that by applying the softmax function to them
# however we do softmax on the attention scores scaled by the square root of the dimension of the key vectors
# this is because whenever the ndimensions increase the softmax function kinda suffers in dividing the probability mass among the many dimensions
# thus will for example give the highest a probability of 0.88 for example and the rest will be very very small numbers
# which will cause gradient instability and thus will make the model hard to train
# due to very small softmax values thus it will be prone to vanishing gradients
# which is why self attention is also called "SCALED DOT PRODUCT ATTENTION"

# finally we compute that context vector Z by taking the weighted sum of the value vectors with the attention scores as weights
# we create the k, q, v vectors by multiplying the input embeddings with learnable weight matrices Wk, Wq, Wv respectively
# this adds a layer of training where the q, k, v would mean different things according the the weights
# thus helps in capturing different aspects of the input and more "queries" as in questions to answer
# for example if we have a sentence like "the cat sat on the mat" and we want to know what is the subject of the sentence
# we can have a query vector that is trained to capture the subject of the sentence and thus
# another head of attention can be trained to capture the object of the sentence and thus we can have multiple heads of attention

# because we are creating a gpt model which is a decoder only model
# we will also need to apply a causal mask to the attention scores to prevent the model from looking ahead in the sequence
# this is done by setting the attention scores of the future tokens to -inf before applying the softmax function
# why not 0? because the softmax function will still give them a non-zero probability and thus the model will still be able to look ahead

# we can also improve computational efficiency by a huge margin by 
# instead of making sequential and distinct queries, keys, and values for each head of attention
# we can make a single query, key, and value for all heads of attention
# and we can instead split the wk, wq, wv weight matrices into multiple heads
# and thus we can have a single matrix multiplication for all heads of attention
# by dividing the dout embedding output dimensions by the number of heads as if we are assigning each head a portion of the output dimensions
# then at the end we can concatenate the outputs of all heads
# then pass them through a final linear layer to get the final output of the multihead attention mechanism which will just "mix" the output
# dimensions all together

In [94]:
# lets start by implementing the multihead attention mechanism from scratch

class MultiHeadAttention(nn.Module):
    def __init__(self, din, dout, n_heads, dropout, context_length, qkv_bias = False):
        super().__init__()

        assert (dout % n_heads == 0), 'Number of heads has to be divisible by the output dimensions'

        self.dropout = nn.Dropout(dropout)
        self.head_dims = dout // n_heads

        # we create the 3 weight matrices the Wq, Wk, Wv
        self.Wq = nn.Linear(din, dout, bias = qkv_bias)
        self.Wk = nn.Linear(din, dout, bias = qkv_bias)
        self.Wv = nn.Linear(din, dout, bias = qkv_bias)

        # will be used for reshaping
        self.n_heads = n_heads
        self.dout = dout

        # we create the mask
        self.register_buffer('mask',
                            torch.triu(torch.ones(context_length, context_length),
                                        diagonal = 1))   # 1 above the main diagonal

        self.output_proj = nn.Linear(dout, dout)

    def forward(self, X):
        n_batches, n_tokens, _ = X.shape

        # now we create the k q v 
        key = self.Wk(X)
        query = self.Wq(X)
        value = self.Wv(X)

        # now each of these tensors have a shape of [n_batches, n_tokens, dout]
        # we want that dout to be instead n_heads, head_dims

        key = key.view(n_batches, n_tokens, self.n_heads, self.head_dims)
        query = query.view(n_batches, n_tokens, self.n_heads, self.head_dims)
        value = value.view(n_batches, n_tokens, self.n_heads, self.head_dims)

        # now we want the n_heads to be before the n_tokens
        # [n_batches, n_heads, n_tokens, head_dims]     # because that would make each head represent a single independant matrix
        # full of tokens as rows where each column represent a dout dimension of that head

        key = key.transpose(1, 2)
        query = query.transpose(1, 2)
        value = value.transpose(1, 2)

        # now we start the full process
        # attention scores W
        attention_scores = query @ key.transpose(-1, -2)

        # we apply the causal mask to the attention_scores
        attention_scores.masked_fill_(self.mask.bool()[:n_tokens, :n_tokens], -torch.inf)   # type: ignore

        # now attention weight A
        attention_weights = torch.softmax(attention_scores / key.shape[-1] ** 0.5, dim = -1)

        # next we apply dropout to the attention weights
        attention_weights = self.dropout(attention_weights)
        
        # now we finally calculate the context vector Z
        context_vectors = attention_weights @ value

        # now we turn them back to [n_batches, n_tokens, dout]
        context_vectors = context_vectors.transpose(1, 2)
        context_vectors = context_vectors.contiguous().view(n_batches, n_tokens, self.dout)

        # finally we pass them onto a final linear layer to capture richer combinations between the inputs
        context_vectors = self.output_proj(context_vectors)

        return context_vectors

In [95]:
# now lets test it out
# we will use the earlier implemented dataloader for this

sample_text = text[:100_000]
tokenizer = tiktoken.get_encoding('gpt2')
dataloader = create_dataloaderV0(sample_text, max_window_length=32, stride=32, batch_size=6,
                                drop_last=True, shuffle=True,
                                tokenizer=tokenizer, num_workers=0)

date_iterator = iter(dataloader)

print(len(dataloader))

116


In [96]:
X, y = next(date_iterator)

In [97]:
X[0]

tensor([   11,   290, 15902,    12,    37, 32275,  7093,    13,   198,  3666,
        21641,   389,   281, 13936,   286, 23168,    11,   393,   355, 18935,
        47958,  1234,   340,    11,   564,   250,   333,   469,   290, 14960,
           11,  1464])

In [98]:
y[0]

tensor([  290, 15902,    12,    37, 32275,  7093,    13,   198,  3666, 21641,
          389,   281, 13936,   286, 23168,    11,   393,   355, 18935, 47958,
         1234,   340,    11,   564,   250,   333,   469,   290, 14960,    11,
         1464,   262])

In [99]:
# as we can see the x is just the y moves 1 index upfront
# this is the sliding window approach in dataloading where we kind of mask the last token and we give it all the previous tokens as input
# this is why it is called self supervised training

In [100]:
# next is embedding them before we use the mha mechanism
# we will represent each token input embedding 768 and context length = 256
context_length = 256
embedding_dim = 768

In [101]:
tok_emb = nn.Embedding(tokenizer.n_vocab, embedding_dim)
pos_emb = nn.Embedding(context_length, embedding_dim)

# now we get the tok_emb and pos_emb for the input X
# the forward of tokemb and posemb is just a simple lookup table where to get the embedding (Weights) for each token
# they just create a sequence of weights (embeddings) per token and the forward of it just returns these weights based on tokenid
batches, tokens = X.shape
tok_embeds = tok_emb(X)
pos_embeds = pos_emb(torch.arange(tokens))

input_embeddings = tok_embeds + pos_embeds

In [102]:
input_embeddings.shape

torch.Size([6, 32, 768])

In [103]:
# now lets test the multihead attention mechanism on these embeddings

mha = MultiHeadAttention(din=embedding_dim, dout=embedding_dim, n_heads=12, dropout=0.1, context_length=context_length)


mha_output = mha(input_embeddings)

In [104]:
mha_output.shape

torch.Size([6, 32, 768])

In [106]:
mha_output[0]

tensor([[ 2.2903e-01, -2.2172e-01, -8.0524e-04,  ...,  8.0775e-01,
          1.5549e-01, -7.0871e-01],
        [ 9.9196e-02,  3.8340e-01,  2.2460e-02,  ...,  6.8666e-01,
          1.6479e-01, -4.1998e-01],
        [ 1.2236e-01,  2.8261e-01,  2.0365e-01,  ...,  4.2657e-01,
         -2.3238e-01, -2.0159e-01],
        ...,
        [ 2.0917e-01, -6.9075e-02,  2.8843e-02,  ..., -4.8239e-02,
          1.3976e-01,  3.1086e-02],
        [ 3.3201e-02, -2.8152e-01, -3.8779e-02,  ..., -1.6240e-01,
         -2.0032e-02, -3.3521e-01],
        [ 1.3791e-01, -2.5111e-01,  4.8979e-02,  ...,  2.5842e-02,
          3.4922e-02, -2.0925e-01]], grad_fn=<SelectBackward0>)